In [1]:
# Cell 1: Verify GPU and install required libraries
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU")

# Install compatible versions that work with Colab's Python 3.12 environment
!pip uninstall -y transformers accelerate datasets -q
!pip install transformers==4.44.2 datasets==2.19.0 accelerate==0.33.0 scikit-learn==1.4.2 numpy==1.26.4 -q

print("\n✅ All libraries installed successfully!")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
GPU Memory: 15.6 GB

✅ All libraries installed successfully!


In [2]:
# Cell 2: Load Financial PhraseBank dataset from HuggingFace
from datasets import load_dataset

# Load the dataset - using sentences where >= 75% annotators agreed on the label
# This gives us higher quality labels for training
dataset = load_dataset(
    "financial_phrasebank",
    "sentences_75agree",
    trust_remote_code=True
)

print(f"Dataset structure: {dataset}")
print(f"\nTotal samples: {len(dataset['train'])}")
print(f"\nFirst 3 examples:")
for i in range(3):
    example = dataset['train'][i]
    label_map = {0: "negative", 1: "neutral", 2: "positive"}
    print(f"  [{label_map[example['label']]}] {example['sentence'][:100]}...")

# Check label distribution
from collections import Counter
labels = [example['label'] for example in dataset['train']]
label_counts = Counter(labels)
label_map = {0: "negative", 1: "neutral", 2: "positive"}
print(f"\nLabel distribution:")
for label_id, count in sorted(label_counts.items()):
    print(f"  {label_map[label_id]}: {count} ({count/len(labels)*100:.1f}%)")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Dataset structure: DatasetDict({
    train: Dataset({
        features: ['sentence', 'label'],
        num_rows: 3453
    })
})

Total samples: 3453

First 3 examples:
  [neutral] According to Gran , the company has no plans to move all production to Russia , although that is whe...
  [positive] With the new production plant the company would increase its capacity to meet the expected increase ...
  [positive] For the last quarter of 2010 , Componenta 's net sales doubled to EUR131m from EUR76m for the same p...

Label distribution:
  negative: 420 (12.2%)
  neutral: 2146 (62.1%)
  positive: 887 (25.7%)


In [3]:
# Cell 3: Split dataset into training (85%) and validation (15%)
from datasets import DatasetDict

# The dataset only has a 'train' split, so we create our own train/val split
# Stratified split preserves the label distribution in both sets
split = dataset['train'].train_test_split(
    test_size=0.15,
    seed=42,
    stratify_by_column='label'
)

dataset_split = DatasetDict({
    'train': split['train'],
    'validation': split['test']
})

print(f"Training samples: {len(dataset_split['train'])}")
print(f"Validation samples: {len(dataset_split['validation'])}")

# Verify stratification preserved label distribution
for split_name in ['train', 'validation']:
    labels = [ex['label'] for ex in dataset_split[split_name]]
    counts = Counter(labels)
    total = len(labels)
    print(f"\n{split_name} distribution:")
    for label_id in sorted(counts.keys()):
        print(f"  {label_map[label_id]}: {counts[label_id]} ({counts[label_id]/total*100:.1f}%)")

Training samples: 2935
Validation samples: 518

train distribution:
  negative: 357 (12.2%)
  neutral: 1824 (62.1%)
  positive: 754 (25.7%)

validation distribution:
  negative: 63 (12.2%)
  neutral: 322 (62.2%)
  positive: 133 (25.7%)


In [4]:
# Cell 4: Load FinBERT tokenizer and tokenize the dataset
from transformers import AutoTokenizer

# Load the pre-trained FinBERT tokenizer
# FinBERT was pre-trained on financial text, so its tokenizer understands
# financial vocabulary (EBITDA, leverage, covenant, etc.) better than generic BERT
model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f"Tokenizer loaded: {model_name}")
print(f"Vocabulary size: {tokenizer.vocab_size}")
print(f"Max sequence length: {tokenizer.model_max_length}")

# Tokenize the dataset
# max_length=512 covers even long financial sentences
# padding="max_length" pads shorter sentences to 512 tokens
# truncation=True cuts sentences longer than 512 tokens
def tokenize_function(examples):
    return tokenizer(
        examples['sentence'],
        padding='max_length',
        truncation=True,
        max_length=512
    )

tokenized_dataset = dataset_split.map(
    tokenize_function,
    batched=True,
    desc="Tokenizing"
)

# Remove the original text column (model only needs token IDs and attention mask)
tokenized_dataset = tokenized_dataset.remove_columns(['sentence'])

# Rename 'label' to 'labels' (what HuggingFace Trainer expects)
tokenized_dataset = tokenized_dataset.rename_column('label', 'labels')

# Set format to PyTorch tensors
tokenized_dataset.set_format('torch')

print(f"\nTokenized dataset ready!")
print(f"Features: {tokenized_dataset['train'].column_names}")
print(f"Sample input_ids shape: {tokenized_dataset['train'][0]['input_ids'].shape}")

Tokenizer loaded: ProsusAI/finbert
Vocabulary size: 30522
Max sequence length: 512

Tokenized dataset ready!
Features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask']
Sample input_ids shape: torch.Size([512])


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [5]:
# Cell 5: Load pre-trained FinBERT with a 3-class classification head
from transformers import AutoModelForSequenceClassification

# num_labels=3 because we have: negative (0), neutral (1), positive (2)
# This loads the pre-trained FinBERT weights and adds a fresh classification layer on top
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size: ~{total_params * 4 / 1e6:.0f} MB (in float32)")

# Move model to GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"\nModel loaded on: {device}")

Total parameters: 109,484,547
Trainable parameters: 109,484,547
Model size: ~438 MB (in float32)

Model loaded on: cuda


In [6]:
# Cell 6: Define evaluation metrics
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report

def compute_metrics(eval_pred):
    """
    This function is called by the HuggingFace Trainer after each evaluation step.
    It receives the model's predictions and the true labels, and returns metrics.
    These metrics appear in the training logs and help us monitor progress.
    """
    logits, labels = eval_pred
    # logits shape: (batch_size, 3) — raw scores for each class
    # We take the class with the highest score as the prediction
    predictions = np.argmax(logits, axis=-1)

    return {
        'accuracy': accuracy_score(labels, predictions),
        'f1_weighted': f1_score(labels, predictions, average='weighted'),
        'f1_macro': f1_score(labels, predictions, average='macro'),
        'precision_weighted': precision_score(labels, predictions, average='weighted'),
        'recall_weighted': recall_score(labels, predictions, average='weighted'),
    }

print("✅ Metrics function defined!")

✅ Metrics function defined!


In [7]:
# Cell 7: Configure training and run fine-tuning
from transformers import TrainingArguments, Trainer

# Training configuration
# These hyperparameters are chosen for FinBERT on Financial PhraseBank
# and will produce a model that integrates cleanly with Agent 3 in Phase 7
training_args = TrainingArguments(
    output_dir='./finbert-sentiment-training',     # Temp directory for checkpoints
    num_train_epochs=5,                             # 5 passes through the training data
    per_device_train_batch_size=16,                 # 16 samples per GPU batch
    per_device_eval_batch_size=32,                  # Larger batch for eval (no gradients = less memory)
    learning_rate=2e-5,                             # Standard fine-tuning LR for BERT-class models
    weight_decay=0.01,                              # L2 regularization to prevent overfitting
    warmup_ratio=0.1,                               # Warm up learning rate for first 10% of steps
    eval_strategy='epoch',                          # Evaluate after each epoch
    save_strategy='epoch',                          # Save checkpoint after each epoch
    load_best_model_at_end=True,                    # After training, load the best checkpoint
    metric_for_best_model='f1_weighted',            # "Best" = highest weighted F1 score
    greater_is_better=True,                         # Higher F1 = better
    logging_dir='./logs',                           # TensorBoard logs
    logging_steps=50,                               # Log metrics every 50 steps
    fp16=True,                                      # Use half-precision for faster training on T4
    seed=42,                                        # Reproducibility
    report_to='none',                               # Don't send metrics to external services
)

# Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    compute_metrics=compute_metrics,
)

# Run training!
print("🚀 Starting FinBERT sentiment fine-tuning...")
print(f"   Training samples: {len(tokenized_dataset['train'])}")
print(f"   Validation samples: {len(tokenized_dataset['validation'])}")
print(f"   Epochs: 5")
print(f"   Batch size: 16")
print(f"   This will take approximately 15-25 minutes on T4 GPU\n")

train_result = trainer.train()

# Print training summary
print(f"\n✅ Training complete!")
print(f"   Total training time: {train_result.metrics['train_runtime']:.0f} seconds")
print(f"   Final training loss: {train_result.metrics['train_loss']:.4f}")

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:488: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


🚀 Starting FinBERT sentiment fine-tuning...
   Training samples: 2935
   Validation samples: 518
   Epochs: 5
   Batch size: 16
   This will take approximately 15-25 minutes on T4 GPU



Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted,F1 Macro,Precision Weighted,Recall Weighted
1,0.305000,0.189488,0.936293,0.936074,0.911771,0.936187,0.936293
2,0.131000,0.166924,0.949807,0.949971,0.933602,0.950271,0.949807
3,0.054700,0.232953,0.944015,0.944607,0.923978,0.945942,0.944015
4,0.014700,0.210483,0.942085,0.942085,0.927227,0.942085,0.942085
5,0.010000,0.212445,0.953668,0.953761,0.943097,0.954093,0.953668



✅ Training complete!
   Total training time: 655 seconds
   Final training loss: 0.2415


In [8]:
# Cell 8: Detailed evaluation on validation set
# Run final evaluation
eval_results = trainer.evaluate()

print("=" * 60)
print("FINBERT SENTIMENT — FINAL EVALUATION RESULTS")
print("=" * 60)
for key, value in eval_results.items():
    if key.startswith('eval_'):
        clean_key = key.replace('eval_', '')
        if isinstance(value, float):
            print(f"  {clean_key:>25s}: {value:.4f}")
        else:
            print(f"  {clean_key:>25s}: {value}")

# Detailed classification report
print("\n" + "=" * 60)
print("DETAILED CLASSIFICATION REPORT")
print("=" * 60)

# Get predictions
predictions_output = trainer.predict(tokenized_dataset['validation'])
predictions = np.argmax(predictions_output.predictions, axis=-1)
true_labels = predictions_output.label_ids

target_names = ['negative', 'neutral', 'positive']
report = classification_report(true_labels, predictions, target_names=target_names, digits=4)
print(report)

FINBERT SENTIMENT — FINAL EVALUATION RESULTS
                       loss: 0.2124
                   accuracy: 0.9537
                f1_weighted: 0.9538
                   f1_macro: 0.9431
         precision_weighted: 0.9541
            recall_weighted: 0.9537
                    runtime: 4.0596
         samples_per_second: 127.6000
           steps_per_second: 4.1880

DETAILED CLASSIFICATION REPORT
              precision    recall  f1-score   support

    negative     0.9206    0.9206    0.9206        63
     neutral     0.9716    0.9565    0.9640       322
    positive     0.9275    0.9624    0.9446       133

    accuracy                         0.9537       518
   macro avg     0.9399    0.9465    0.9431       518
weighted avg     0.9541    0.9537    0.9538       518



In [9]:
# Cell 9: Save the fine-tuned model to Google Drive
# We save to Google Drive so the model persists even after Colab disconnects

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Create directory for model files
import os
save_dir = '/content/drive/MyDrive/finsight-ai-models/finbert-sentiment'
os.makedirs(save_dir, exist_ok=True)

# Save the model and tokenizer
# This saves: config.json, model.safetensors (weights), tokenizer files
# These are the EXACT files that Agent 3 will load in Phase 7
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

# Verify saved files
print("✅ Model saved to Google Drive!")
print(f"   Location: {save_dir}")
print(f"\n   Saved files:")
for f in sorted(os.listdir(save_dir)):
    size_mb = os.path.getsize(os.path.join(save_dir, f)) / 1e6
    print(f"   {f:40s} ({size_mb:.1f} MB)")

Mounted at /content/drive
✅ Model saved to Google Drive!
   Location: /content/drive/MyDrive/finsight-ai-models/finbert-sentiment

   Saved files:
   config.json                              (0.0 MB)
   model.safetensors                        (438.0 MB)
   special_tokens_map.json                  (0.0 MB)
   tokenizer.json                           (0.7 MB)
   tokenizer_config.json                    (0.0 MB)
   training_args.bin                        (0.0 MB)
   vocab.txt                                (0.2 MB)


In [10]:
# Cell 10: Save evaluation metrics as JSON for project documentation
# These metrics go into README.md and are referenced in interviews
import json

metrics_to_save = {
    "model": "ProsusAI/finbert",
    "task": "financial_sentiment_classification",
    "num_classes": 3,
    "class_names": ["negative", "neutral", "positive"],
    "dataset": "financial_phrasebank_75agree",
    "train_samples": len(tokenized_dataset['train']),
    "val_samples": len(tokenized_dataset['validation']),
    "epochs": 5,
    "learning_rate": 2e-5,
    "batch_size": 16,
    "eval_results": {k.replace('eval_', ''): round(v, 4) if isinstance(v, float) else v
                     for k, v in eval_results.items()},
    "training_time_seconds": round(train_result.metrics['train_runtime'], 0),
}

metrics_path = '/content/drive/MyDrive/finsight-ai-models/finbert-sentiment/training_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics_to_save, f, indent=2)

print("✅ Training metrics saved!")
print(json.dumps(metrics_to_save, indent=2))

✅ Training metrics saved!
{
  "model": "ProsusAI/finbert",
  "task": "financial_sentiment_classification",
  "num_classes": 3,
  "class_names": [
    "negative",
    "neutral",
    "positive"
  ],
  "dataset": "financial_phrasebank_75agree",
  "train_samples": 2935,
  "val_samples": 518,
  "epochs": 5,
  "learning_rate": 2e-05,
  "batch_size": 16,
  "eval_results": {
    "loss": 0.2124,
    "accuracy": 0.9537,
    "f1_weighted": 0.9538,
    "f1_macro": 0.9431,
    "precision_weighted": 0.9541,
    "recall_weighted": 0.9537,
    "runtime": 4.0596,
    "samples_per_second": 127.6,
    "steps_per_second": 4.188,
    "epoch": 5.0
  },
  "training_time_seconds": 655.0
}
